
# Silver (ruwe data transformeren)
_Data transformeren en wegschrijven naar silver tables. 
Vanuit daar kan het gebruikt worden voor de gold layer._

In [0]:
# Module imports
import pandas as pd
import glob
import shutil
import os

In [0]:
# CSV inladen vanuit bronze map
files = glob.glob("/Volumes/workspace/bronze/bronze/offline_coingecko_prices_*.csv")

# Alle CSV bestanden samenvoegen in 1 dataframe
df = pd.concat([pd.read_csv(f) for f in files])

# Data bekijken
df.info()
df.head()

In [0]:
# Opschonen & normaliseren

df_silver = df.copy()

# Normaliseren
df_silver["coin_id"] = df_silver["coin_id"].str.lower() # LowerCase
df_silver["price"] = pd.to_numeric(df_silver["price"], errors="coerce").round(2) # NaN als geen getal + afronding
df_silver["price_timestamp"] = pd.to_datetime(df_silver["price_timestamp"]) # omzetten naar datetime

# Lege waardes verwijderen op basis van coin_id, price_timestamp en price
df_silver = df_silver.dropna(subset=["coin_id", "price_timestamp", "price"])

# Duplicaten verwijderen op basis van coin_id en price_timestamp
df_silver = df_silver.drop_duplicates(subset=["coin_id", "price_timestamp"])

# Extra kolom
df_silver["date"] = df_silver["price_timestamp"].dt.date # datum kolom maken

df_silver.head()

In [0]:
# Silver tabel lezen
existing_df = spark.table("silver.crypto_prices")

# Dataframe met nieuwe data
df_new = spark.createDataFrame(df_silver)

# Filteren op basis van coin_id en price_timestamp
df_filtered = df_new.join(
    existing_df,
    on=["coin_id", "price_timestamp"],
    how="left_anti"   # alleen nieuwe records
)

# Wegschrijven nieuwe data naar silver
df_filtered.write.mode("append").saveAsTable("silver.crypto_prices")

In [0]:
# Verwerkte CSV bestand naar map verwerkt plaatsen

# bronze map pad
bronze_dir = "/Volumes/workspace/bronze/bronze"

# lijst maken van verwerkte bestanden
csv_files = glob.glob(f"{bronze_dir}/*.csv")

# verwerkt map aanmaken
verwerkt_dir = "/Volumes/workspace/bronze/bronze/verwerkt"
os.makedirs(verwerkt_dir, exist_ok=True)

# bestanden verplaatsen met for loop
for file in csv_files:
    filename = os.path.basename(file)
    new_path = f"{verwerkt_dir}/{filename}"

    print(f"Verplaatsen: {file} → {new_path}")
    shutil.move(file, new_path)
